# ESTA ES LA VERSIÓN DE DIAGNÓSTICO VERBOSO

**Versión:** `v0.8-websearch-debug`  
**Última modificación:** `2026-09-11 20:05 CEST`  
**Rama:** `fix/chapter2-news-agent`  
**Modelo:** `gpt-4.1-mini`

Esta versión añade una prueba directa de Responses API, otra del Agents SDK y un flujo Evaluator-Optimizer con `raw_responses` visible.


## 1. Instalar dependencias

In [7]:
!pip install -U openai openai-agents -q

## 2. Imports y entorno

In [8]:
import json, os, sys, importlib.metadata as im
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Literal
from urllib.parse import urlsplit
from google.colab import userdata
from openai import OpenAI
from agents import Agent, Runner, WebSearchTool, ModelSettings, ItemHelpers

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("Python:", sys.version)
print("openai:", im.version("openai"))
print("openai-agents:", im.version("openai-agents"))
print("API key:", bool(os.environ.get("OPENAI_API_KEY")))

def dump(obj):
    return obj.model_dump() if hasattr(obj, "model_dump") else obj

def urls_in(obj):
    if hasattr(obj, "model_dump"): obj=obj.model_dump()
    out=[]
    if isinstance(obj, dict):
        for k,v in obj.items():
            if k=="url" and isinstance(v,str) and v.startswith("http"): out.append(v)
            out += urls_in(v)
    elif isinstance(obj,(list,tuple)):
        for v in obj: out += urls_in(v)
    return list(dict.fromkeys(out))

def show(label, obj):
    print("\n"+"="*25, label, "="*25)
    d=dump(obj)
    print(json.dumps(d, indent=2, default=str)[:30000])
    us=urls_in(obj)
    print("\nURLs:", len(us))
    for u in us[:50]: print(" ",u)
    return us


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
openai: 3.13.0
openai-agents: 0.22.2
API key: True


## 3. PRUEBA A — Responses API directa (sin Agents SDK)

In [9]:
client=OpenAI()
direct = client.responses.create(
    model="gpt-4.1-mini",
    tools=[{"type":"web_search","external_web_access":True}],
    tool_choice="required",
    include=["web_search_call.action.sources"],
    input="Search the live web. Find ONE Reuters article published on September 10 or 11, 2026 about AI, OpenAI, Nvidia, Oracle, Adobe, AI infrastructure or AI investment. Cite it."
)
print("OUTPUT TEXT:\n", direct.output_text)
print("OUTPUT TYPES:", [getattr(x,"type",None) for x in direct.output])
show("DIRECT RESPONSE RAW", direct)


OUTPUT TEXT:
 I searched for a Reuters article published on September 10 or 11, 2026, covering topics such as AI, OpenAI, Nvidia, Oracle, Adobe, AI infrastructure, or AI investment. Unfortunately, I couldn't locate any relevant articles from Reuters within that timeframe. It's possible that such articles haven't been published yet or are not readily accessible. If you have any other questions or need information on a different topic, feel free to ask. 
OUTPUT TYPES: ['web_search_call', 'message']

========================= DIRECT RESPONSE RAW =========================
{
  "id": "resp_02add9567776ae58006aa443d28a3087d0b28309b85c2349c9",
  "created_at": 1789150162.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4.1-mini-2025-04-14",
  "object": "response",
  "output": [
    {
      "id": "ws_02add9567776ae58006aa443d4c89087d08642e6cc66fca5af",
      "action": {
        "type": "search",
        "queries": [
          "Search the

[]

## 4. PRUEBA B — Agents SDK + WebSearchTool

In [10]:
probe=Agent(
    name="probe",
    model="gpt-4.1-mini",
    instructions="You MUST use web search. Find one Reuters article from September 10 or 11, 2026 about AI. Cite it. Do not answer from memory.",
    tools=[WebSearchTool(search_context_size="high", external_web_access=True)],
    model_settings=ModelSettings(tool_choice="required",response_include=["web_search_call.action.sources"])
)
pr=await Runner.run(probe,"Search now.")
print("FINAL OUTPUT:\n",pr.final_output)
print("NEW ITEM CLASSES:",[type(x).__name__ for x in pr.new_items])
print("RAW ITEM TYPES:",[getattr(getattr(x,"raw_item",None),"type",None) for x in pr.new_items])
print("RAW RESPONSES:",len(pr.raw_responses))
for i,r in enumerate(pr.raw_responses):
    show(f"AGENTS RAW RESPONSE {i}",r)


FINAL OUTPUT:
 I searched for a Reuters article from September 10 or 11, 2026, about AI, but I couldn't find any results. It's possible that no such article was published during that time frame. If you have a specific topic or question about AI, feel free to ask, and I'll be glad to assist you. 
NEW ITEM CLASSES: ['ToolCallItem', 'MessageOutputItem']
RAW ITEM TYPES: ['web_search_call', 'message']
RAW RESPONSES: 1

========================= AGENTS RAW RESPONSE 0 =========================
"ModelResponse(output=[ResponseFunctionWebSearch(id='ws_009a81b9af08e2b5006aa443df7afc87d0bb123b5a3dcf76c4', action=ActionSearch(type='search', queries=['Search now.'], query='Search now.', sources=None), status='completed', type='web_search_call'), ResponseOutputMessage(id='msg_009a81b9af08e2b5006aa443e0698887d0bb640651dc447039', content=[ResponseOutputText(annotations=[], text=\"I searched for a Reuters article from September 10 or 11, 2026, about AI, but I couldn't find any results. It's possible tha

## 5. Flujo Evaluator-Optimizer con diagnóstico

In [11]:
today=datetime.now().strftime("%Y-%m-%d")
start=(datetime.now()-timedelta(days=2)).strftime("%Y-%m-%d")

searcher=Agent(
    name="web_news_searcher",
    model="gpt-4.1-mini",
    instructions=f'''You MUST use web search and never answer from memory.
Find genuine Reuters articles between {start} and {today}.
For each item provide headline, date, summary and inline citation.
Search requested companies/topics separately if needed. Do not fabricate.''',
    tools=[WebSearchTool(search_context_size="high",external_web_access=True)],
    model_settings=ModelSettings(tool_choice="required",response_include=["web_search_call.action.sources"])
)

@dataclass
class Eval:
    feedback:str
    score:Literal["successful","unsuccessful"]

evaluator=Agent(
    name="evaluator",
    model="gpt-4.1-mini",
    instructions=f'''Strictly evaluate the original request. Successful only if the requested number of Reuters articles is present, each with headline/date/summary, dates between {start} and {today}, and genuine Reuters URL evidence. Zero results is always unsuccessful.''',
    output_type=Eval
)

async def main():
    q=input("User's request: " ).strip()
    feedback=None
    last=""
    for n in range(1,3):
        prompt=q if feedback is None else q+"\nPrevious failure: "+feedback+"\nRun a NEW web search."
        r=await Runner.run(searcher,prompt)
        last=ItemHelpers.text_message_outputs(r.new_items)
        print(f"\n***** NEWS SEARCH {n} *****\n{last}")
        print("new_items:",[(type(x).__name__,getattr(getattr(x,"raw_item",None),"type",None)) for x in r.new_items])
        allurls=[]
        for rr in r.raw_responses:
            allurls += urls_in(rr)
        allurls=list(dict.fromkeys(allurls))
        reuters=[u for u in allurls if "reuters.com" in urlsplit(u).netloc.lower()]
        print("RAW RESPONSES:",len(r.raw_responses))
        print("ALL URLS:",len(allurls))
        for u in allurls[:50]: print(" SOURCE:",u)
        print("REUTERS URLS:",len(reuters))
        for u in reuters: print(" REUTERS:",u)
        for i,rr in enumerate(r.raw_responses):
            print(f"\n--- RAW RESPONSE {i} (first 30000 chars) ---")
            print(json.dumps(dump(rr),indent=2,default=str)[:30000])
        ev=await Runner.run(evaluator,f"ORIGINAL REQUEST:\n{q}\n\nANSWER:\n{last}\n\nREUTERS URLS:\n"+("\n".join(reuters) if reuters else "NONE"))
        res:Eval=ev.final_output
        print("EVALUATOR:",res.score,res.feedback)
        if res.score=="successful": break
        feedback=res.feedback
    print("\nFINAL:\n",last)


## 6. Ejecutar

In [12]:
await main()

User's request: Noticias de nvidia en reuters

***** NEWS SEARCH 1 *****
No se encontraron noticias recientes sobre Nvidia en Reuters entre el 9 y el 11 de septiembre de 2026. Es posible que no haya habido desarrollos significativos en ese período. Para obtener información más actualizada, le recomiendo visitar el sitio web oficial de Nvidia o consultar otras fuentes de noticias financieras. 
new_items: [('ToolCallItem', 'web_search_call'), ('MessageOutputItem', 'message')]
RAW RESPONSES: 1
ALL URLS: 0
REUTERS URLS: 0

--- RAW RESPONSE 0 (first 30000 chars) ---
"ModelResponse(output=[ResponseFunctionWebSearch(id='ws_0b5507b9f7f24b72006aa443fd4a6487d098401008236161d0', action=ActionSearch(type='search', queries=['Noticias de nvidia en reuters'], query='Noticias de nvidia en reuters', sources=None), status='completed', type='web_search_call'), ResponseOutputMessage(id='msg_0b5507b9f7f24b72006aa443fe489c87d08962e1efa30c2c13', content=[ResponseOutputText(annotations=[], text='No se encontr